[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nammaschool/interactive-learning/blob/main/notebooks/courses/complex_numbers_course.ipynb)

# Complex numbers full course: practice notebook

One practice question for every teaching chapter of the video, in the same order, and two pictures you can move
yourself: the powers of i, turning a quarter at a time, and two arrows whose product turns and stretches as you move
them. Work each question out on paper first, then type your answer into the box and press **check**. It tells you
whether you're right. Press **show working** if you'd like to see the steps.

**How to use this notebook.** The grey boxes are cells. Click the first one and press **Shift + Enter** (or the ▶ Run
button) to run it. Run each cell in turn, from the top down. When sliders appear, move them and watch. Change any
number and run the cell again: nothing can break. It works in your browser (no install) and in Google Colab.

**Typing answers.** Write i as `i`: `3 - 2i`, `7/13 + 17/13 i`. Roots as `sqrt(3)`, π as `pi`, minus as `-`. Angles
in degrees as plain numbers (`95`); angles in radians with pi (`5pi/6`). For several answers, separate them with
commas, in the order the question asks. When a question asks for an equation, type it in any form you like, such
as `x^2 + 4x + 29 = 0`. You can also put your answer between the quotes in a cell's last line, `practice(1, "")`,
and run the cell: it checks straight away.

**The cheat sheet.** Every result from the course on two pages:
[complex numbers cheat sheet (PDF)](https://nammaschool.github.io/interactive-learning/cheatsheets/complex_numbers_cheat_sheet.pdf).

In [ ]:
# Setup: in the browser version (JupyterLite) the sliders need one install; elsewhere this does nothing.
import sys
if "pyodide" in sys.modules:
    import piplite
    await piplite.install(["ipywidgets"])

In [ ]:
# The look and the checker: run this cell first.
import math, cmath, re
from itertools import permutations
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from sympy.parsing.sympy_parser import parse_expr, standard_transformations, implicit_multiplication_application, convert_xor
from IPython.display import display
from ipywidgets import interact, IntSlider, FloatSlider, Text, Button, Output, HBox, VBox, Layout

SLATE, SLATE2 = "#1f2b28", "#2a3a36"        # the slate ground
CHALK, DIM, GHOST = "#ece6d6", "#93a097", "#3b4c47"   # chalk, chalk smudge, ghost lines
CORAL = "#e3896b"                           # the one key thing in each figure
SKY, MINT, GOLD = "#6fb6bb", "#a0c27c", "#e8c56b"     # the first arrow · the second arrow · the answer

plt.rcParams.update({
    "figure.facecolor": SLATE, "axes.facecolor": SLATE, "savefig.facecolor": SLATE,
    "text.color": CHALK, "axes.labelcolor": CHALK, "axes.edgecolor": DIM, "axes.titlecolor": CHALK,
    "xtick.color": DIM, "ytick.color": DIM, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GHOST, "grid.linewidth": 0.8,
    "font.size": 13, "axes.titlesize": 15, "axes.labelsize": 13, "legend.fontsize": 12,
    "lines.linewidth": 2.4, "figure.dpi": 100,
})

RULES = standard_transformations + (implicit_multiplication_application, convert_xor)
x = sp.Symbol("x")
I, pi, sqrt = sp.I, sp.pi, sp.sqrt
NAMES = {"i": I, "j": I, "pi": pi, "e": sp.E, "sqrt": sqrt, "cos": sp.cos, "sin": sp.sin, "exp": sp.exp, "x": x}

def prep(s):
    """Typed symbols → Python: − × ÷ ° ² ³ √ π."""
    s = (s.replace("−", "-").replace("×", "*").replace("÷", "/").replace("°", "").replace("²", "^2")
          .replace("³", "^3").replace("π", "pi"))
    return re.sub(r"√\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", s).replace("√", "sqrt")   # √3 → sqrt(3); √(…) stays

def read_one(p):
    """One typed answer: a number, or an equation in x (everything moved to one side, so it equals zero)."""
    if "=" in p:
        left, right = p.replace("==", "=").split("=", 1)
        return sp.expand(parse_expr(left, local_dict=NAMES, transformations=RULES)
                         - parse_expr(right, local_dict=NAMES, transformations=RULES))
    return parse_expr(p, local_dict=NAMES, transformations=RULES)

def read(s):
    """Your typed answer (a list, one for each answer)."""
    return [read_one(p) for p in prep(s).split(",") if p.strip()]

def same_equation(got, want):
    """Two equations in x (each as 'something = 0') are the same: one is a number times the other."""
    try:
        got, want = sp.expand(sp.sympify(got)), sp.expand(want)
        if got == 0 or not got.free_symbols <= {x}: return False
        cw, cg = sp.Poly(want, x).LC(), sp.Poly(got, x).coeff_monomial(x ** sp.degree(want, x))
        return cg != 0 and sp.expand(got * cw - want * cg) == 0
    except Exception: return False

def same(got, want, tol=1e-9):
    if sp.sympify(want).free_symbols: return same_equation(got, want)     # an equation
    try: return abs(complex(sp.N(sp.sympify(got) - want, 30))) < tol
    except (TypeError, ValueError): return False                         # a letter, not a number

def cform(z, d=2):
    """A complex number as a + bi, rounded: 0.87 + 0.50i, −1.00 − 0.00i becomes −1.00."""
    a, b = round(z.real, d) + 0.0, round(z.imag, d) + 0.0
    if b == 0: return f"{a:.{d}f}".replace("-", "−")
    return f"{a:.{d}f} {'+' if b > 0 else '−'} {abs(b):.{d}f}i".replace("-", "−")

def cis(deg): return complex(math.cos(math.radians(deg)), math.sin(math.radians(deg)))

def plane(ax, lim):
    """The Argand plane: the real axis across, the imaginary axis up, one unit the same both ways."""
    ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim); ax.set_aspect("equal")
    ax.axhline(0, color=DIM, lw=1.4); ax.axvline(0, color=DIM, lw=1.4)
    ax.text(lim - 0.03 * lim, -0.05 * lim, "real", color=DIM, ha="right", va="top", fontsize=12)
    ax.text(0.05 * lim, lim - 0.05 * lim, "imaginary", color=DIM, va="top", fontsize=12)

def arrow(ax, q, col=CHALK, lw=3.2, head=24, alpha=1.0, z=5):
    """A number as an arrow from 0 to the point q (a Python complex number)."""
    ax.annotate("", xy=(q.real, q.imag), xytext=(0, 0), zorder=z,
                arrowprops=dict(arrowstyle="-|>", color=col, lw=lw, mutation_scale=head, alpha=alpha,
                                shrinkA=0, shrinkB=0))

def arc(ax, r, a0, a1, col, lw=2.2, alpha=1.0):
    """An angle mark from a0 to a1 degrees at radius r."""
    t = np.radians(np.linspace(a0, a1, max(2, int(abs(a1 - a0)) + 1)))
    ax.plot(r * np.cos(t), r * np.sin(t), color=col, lw=lw, alpha=alpha)

# Each question: the question, the answers in order, the working, (for rounded answers) how close counts, (for answers
# that can come in any order) which ones, and (if it has several parts) the move that cracks each part, shown when that
# part is wrong.
Q = {}

def marks(got, q):
    """Right or wrong for each typed answer (answers in an any-order group may come in either order)."""
    want, tol = q["answer"], q.get("tol", [1e-9] * len(q["answer"]))
    ok = [same(g, w, t) for g, w, t in zip(got, want, tol)]
    for grp in q.get("any_order", []):
        if all(ok[k] for k in grp): continue
        for p in permutations(grp):
            if all(same(got[k], want[j], tol[j]) for k, j in zip(grp, p)):
                for k in grp: ok[k] = True
                break
    return ok

def practice(n, answer=""):
    """Question n: an answer box, a check button and a show-working button."""
    q = Q[n]
    box = Text(value=answer, placeholder="your answer", description="answer:", layout=Layout(width="460px"))
    check, work, out = Button(description="check"), Button(description="show working"), Output()
    def on_check(_=None):
        out.clear_output()
        with out:
            try: got = read(box.value)
            except Exception: print("I can't read that yet: check the brackets and the signs."); return
            want = q["answer"]
            if len(got) != len(want): print(f"✗ the question asks for {len(want)} answer(s); you typed {len(got)}."); return
            ok = marks(got, q)
            if all(ok): print("✓ right"); return
            if "moves" not in q: print("✗ not quite: try again, or press show working"); return
            for k, (good, move) in enumerate(zip(ok, q["moves"]), 1):
                print(f"✓ part {k}: right" if good else f"✗ part {k}: not quite. The move that cracks it: {move}")
    def on_work(_=None):
        out.clear_output()
        with out:
            for line in q["working"]: print(line)
    check.on_click(on_check); work.on_click(on_work)
    print(f"Question {n}. {q['text']}")
    display(VBox([HBox([box, check, work]), out]))
    if answer: on_check()

## 1 · The imaginary unit i

Draw each number as an arrow from zero. Multiplying by −1 is a half turn, and i is a quarter turn: done twice, it
makes the half turn, so i² = −1. Each power of i turns one more quarter: i, −1, −i, 1, and then it repeats every
four. For a big power, take out the full turns: i³⁵ = i³ = −i, because 35 = 4 × 8 + 3. And √−9 = 3i, but be careful:
√a × √b = √(ab) only when a and b aren't negative, so write each root as a multiple of i first.

Move the slider: the arrow turns a quarter for each step, and the table of powers fills in beside it.

In [ ]:
VALUES = {0: "1", 1: "i", 2: "−1", 3: "−i"}            # i to the n, by the remainder when n is divided by 4
SUP = str.maketrans("0123456789-", "⁰¹²³⁴⁵⁶⁷⁸⁹⁻")
def sup(n): return str(n).translate(SUP)
def power(n): return VALUES[n % 4]                    # Python's % gives the remainder 0–3, even for negative n

assert all(I ** n == [1, I, -1, -I][n % 4] for n in range(-12, 120))       # checked against sympy
assert (power(35), power(100), power(-7), power(-1)) == ("−i", "1", "i", "−i")   # the video's examples
assert sum(I ** n for n in range(1, 5)) == 0                               # any four in a row add to zero

def powers(n=1):
    fig = plt.figure(figsize=(11.5, 4.9))
    ax = fig.add_axes([0.02, 0.05, 0.38, 0.9]); plane(ax, 1.75); ax.set_xticks([]); ax.set_yticks([])
    t = np.linspace(0, 2 * math.pi, 200); ax.plot(np.cos(t), np.sin(t), color=GHOST, lw=1.6, ls="--")
    for sp_ in ("left", "bottom"): ax.spines[sp_].set_visible(False)
    for lab, (lx, ly) in zip(VALUES.values(), ((1.28, 0.15), (0.16, 1.3), (-1.3, 0.15), (0.2, -1.3))):
        ax.text(lx, ly, lab, color=DIM, fontsize=15, ha="center", va="center")   # the four places an arrow can point
    for k in range(1, n + 1):                                                 # one quarter-turn arc per step
        arc(ax, 0.22 + 0.045 * k, 90 * (k - 1) + 4, 90 * k - 4, GOLD, lw=2.2 if k == n else 1.4,
            alpha=1.0 if k == n else 0.45)
    arrow(ax, cis(90 * n), CORAL, lw=4)
    p = cis(90 * n)
    ax.plot([p.real], [p.imag], "o", color=CORAL, ms=9, zorder=6)
    tb = fig.add_axes([0.44, 0.0, 0.56, 1.0]); tb.axis("off"); tb.set_xlim(0, 1); tb.set_ylim(0, 1)
    tb.text(0.0, 0.93, "the powers of i", color=DIM, fontsize=15)
    for c, lab in enumerate(("up", "left", "down", "right")):
        tb.text(0.02 + 0.245 * c, 0.84, lab, color=GHOST, fontsize=13)
    for k in range(1, min(n, 16) + 1):                                        # fills in, four to a row
        row, col = (k - 1) // 4, (k - 1) % 4
        tb.text(0.02 + 0.245 * col, 0.74 - 0.11 * row, f"i{sup(k)} = {power(k)}", family="monospace",
                fontsize=19 if k == n else 17, color=CORAL if k == n else CHALK,
                fontweight="bold" if k == n else "normal")
    q, r = divmod(n, 4)
    msg = ("i⁰ = 1: no turn at all" if n == 0 else
           f"{n} = 4 × {q} + {r}, so i{sup(n)} = i{sup(r)} = {power(n)}" if q else
           f"{n} quarter turn{'s' if n > 1 else ''}: i{sup(n)} = {power(n)}")
    tb.text(0.0, 0.18, msg, color=GOLD, fontsize=18, fontweight="bold")
    tb.text(0.0, 0.07, "every four steps is a full turn, so the pattern repeats", color=DIM, fontsize=14)
    plt.show()

In [ ]:
interact(powers, n=IntSlider(value=1, min=0, max=16, description="n"));

In [ ]:
Q[1] = {"text": "Work out i to the 47 and i to the −3. Then work out √−25 × √−4.",
        "answer": [I ** 47, I ** -3, (5 * I) * (2 * I)],
        "moves": ["take out full turns: 47 = 4 × 11 + 3.",
                  "−3 = 4 × (−1) + 1: one full turn backwards, then the remainder.",
                  "write each root as a multiple of i first: √−25 = 5i and √−4 = 2i."],
        "working": ["i⁴⁷: 47 = 4 × 11 + 3. Eleven full turns do nothing, so i⁴⁷ = i³ = −i.",
                    "i⁻³: −3 = 4 × (−1) + 1, so i⁻³ = i¹ = i. (Check: i⁻³ × i³ = i × (−i) = −i² = 1.)",
                    "√−25 = 5i and √−4 = 2i, so the product is 5i × 2i = 10i² = −10.",
                    "Not √100 = 10: √a × √b = √(ab) only when a and b aren't negative."]}
assert Q[1]["answer"] == [-I, I, -10] and sqrt(-25) * sqrt(-4) == -10 and 47 == 4 * 11 + 3
practice(1)

## 2 · Complex numbers and their algebra

A complex number is z = a + bi, with real part a and imaginary part b (the b alone, without the i). Two complex
numbers are equal when their real parts match and their imaginary parts match. Add and subtract part by part.
Multiply by expanding the brackets, with i² = −1. The conjugate of z = a + bi is z̄ = a − bi, and z z̄ = a² + b², a
real number. To divide, multiply the top and the bottom by the conjugate of the bottom.

In [ ]:
Q[2] = {"text": "Work out (2 + 3i)(1 − 4i). Then work out (5 + i) ÷ (2 − 3i) in the form a + bi.",
        "answer": [sp.expand((2 + 3 * I) * (1 - 4 * I)), sp.expand((5 + I) * (2 + 3 * I) / 13)],
        "moves": ["expand: 2 × 1, 2 × (−4i), 3i × 1, 3i × (−4i), and remember i² = −1.",
                  "multiply the top and the bottom by 2 + 3i, the conjugate of the bottom."],
        "working": ["(2 + 3i)(1 − 4i) = 2 − 8i + 3i − 12i².",
                    "−12i² = +12, so it's 2 + 12 − 5i = 14 − 5i.",
                    "(5 + i) ÷ (2 − 3i): multiply the top and the bottom by 2 + 3i.",
                    "Bottom: (2 − 3i)(2 + 3i) = 4 + 9 = 13.   Top: 10 + 15i + 2i + 3i² = 7 + 17i.",
                    "So the answer is 7/13 + (17/13)i.",
                    "Check: (2 − 3i)(7/13 + 17/13 i) = (14 + 51)/13 + (34 − 21)/13 i = 5 + i."]}
assert Q[2]["answer"] == [14 - 5 * I, sp.Rational(7, 13) + sp.Rational(17, 13) * I]
assert sp.expand((2 - 3 * I) * Q[2]["answer"][1]) == 5 + I
practice(2)

## 3 · The Argand plane and the modulus

Put the real numbers across and the imaginary numbers up, and every complex number a + bi is a point, or an arrow
from zero. Adding is arrows nose to tail. The modulus |z| = √(a² + b²) is the arrow's length (Pythagoras), and
z z̄ = |z|². |z₁ − z₂| is the distance between two points. Moduli multiply: |z₁z₂| = |z₁||z₂|, and a sum is never
longer than its parts: |z₁ + z₂| ≤ |z₁| + |z₂|.

In [ ]:
Q[3] = {"text": "Find |−6 + 8i|. Then find the distance between the points 2 + i and −1 + 5i. Then find "
                "|(3 + 4i)(5 − 12i)| without multiplying out the brackets.",
        "answer": [sp.Abs(-6 + 8 * I), sp.Abs((-1 + 5 * I) - (2 + I)), sp.Abs(3 + 4 * I) * sp.Abs(5 - 12 * I)],
        "moves": ["the square root of 6² + 8².",
                  "the distance is the modulus of the difference: (−1 + 5i) − (2 + i).",
                  "moduli multiply: |3 + 4i| × |5 − 12i|."],
        "working": ["|−6 + 8i| = √(36 + 64) = √100 = 10.",
                    "(−1 + 5i) − (2 + i) = −3 + 4i, and |−3 + 4i| = √(9 + 16) = 5. They're 5 apart.",
                    "|3 + 4i| = 5 and |5 − 12i| = √(25 + 144) = 13, so |(3 + 4i)(5 − 12i)| = 5 × 13 = 65.",
                    "Check: (3 + 4i)(5 − 12i) = 63 − 16i, and √(63² + 16²) = √4225 = 65."]}
assert Q[3]["answer"] == [10, 5, 65] and sp.expand((3 + 4 * I) * (5 - 12 * I)) == 63 - 16 * I and 63 ** 2 + 16 ** 2 == 65 ** 2
practice(3)

## 4 · Polar form

The argument, arg z, is the angle the arrow makes with the positive real axis, measured anticlockwise. Its tangent
is b/a, but check which quarter of the plane the point is in: 1 + i and −1 − i have the same tangent. The principal
argument lies between −π and π. With r = |z| and θ = arg z, a = r cos θ and b = r sin θ, so z = r(cos θ + i sin θ).
And |z − z₀| = r is a circle with centre z₀ and radius r, a locus.

In [ ]:
Q[4] = {"text": "Write −√3 + i in polar form: give its modulus and its principal argument (in radians, like 5pi/6). "
                "Then z moves on the circle |z − 3 − 4i| = 2: find the smallest value |z| can take.",
        "answer": [sp.Abs(-sqrt(3) + I), sp.arg(-sqrt(3) + I), sp.Abs(3 + 4 * I) - 2],
        "moves": ["the square root of (√3)² + 1².",
                  "tan θ = 1/(−√3), and the point is up and to the left, so θ is between π/2 and π.",
                  "the centre 3 + 4i is |3 + 4i| from zero; the nearest point of the circle is one radius closer."],
        "working": ["Modulus: √(3 + 1) = √4 = 2.",
                    "The point is up and to the left. Its tangent is 1/(−√3), and the angle with tangent 1/√3 is π/6,",
                    "so the argument is π − π/6 = 5π/6 (150°).",
                    "So −√3 + i = 2(cos 5π/6 + i sin 5π/6). Check: 2 cos 5π/6 = −√3 and 2 sin 5π/6 = 1.",
                    "The circle has centre 3 + 4i and radius 2. The centre is √(9 + 16) = 5 from zero,",
                    "so the nearest point of the circle is 5 − 2 = 3 from zero."]}
assert Q[4]["answer"] == [2, 5 * pi / 6, 3]
assert sp.simplify(2 * sp.cos(5 * pi / 6) + sqrt(3)) == 0 and sp.simplify(2 * sp.sin(5 * pi / 6)) == 1
practice(4)

## 5 · Multiplying and dividing in polar form

Multiplying by i turns an arrow a quarter; multiplying by 2 stretches it. A general complex number does both: to
multiply, multiply the lengths and add the angles, r₁(cos θ₁ + i sin θ₁) × r₂(cos θ₂ + i sin θ₂) =
r₁r₂(cos(θ₁ + θ₂) + i sin(θ₁ + θ₂)). To divide, divide the lengths and subtract the angles.

Move the two arrows with their sliders, z₁ (a length r₁ and an angle θ₁) and z₂: the third arrow is their product,
turning and stretching as they move. Can you make the product land exactly on −1, the gold ring?

In [ ]:
def principal(a):
    """An angle in degrees as the principal argument, between −180° and 180° (180° included)."""
    a = (a + 180) % 360 - 180
    return 180 if a == -180 else a

def length(r): return f"{r:g}"
def ang(a): return f"{a}°" if a >= 0 else f"(−{-a}°)"                 # an angle in degrees, a negative one bracketed
LIM = 3.3                                                              # the picture runs from −3.3 to 3.3

for r1_, t1_, r2_, t2_ in ((1.5, 30, 2, 45), (0.75, -120, 1.25, 165), (2, 170, 2, 170)):      # the rule, checked
    assert abs(r1_ * cis(t1_) * r2_ * cis(t2_) - r1_ * r2_ * cis(t1_ + t2_)) < 1e-12
assert abs((3 + 1j) * (1 + 1j) - (2 + 4j)) < 1e-12                                  # the video's example:
assert round(math.degrees(cmath.phase(3 + 1j)), 1) == 18.4 and round(math.degrees(cmath.phase(2 + 4j)), 1) == 63.4
assert abs(abs(3 + 1j) * abs(1 + 1j) - abs(2 + 4j)) < 1e-12                        # √10 × √2 = √20, 18.4° + 45° = 63.4°
assert abs(1 * cis(90) * 1 * cis(90) + 1) < 1e-12 and principal(270) == -90 and principal(-180) == 180   # i × i = −1

def product(r1=1.0, t1=30, r2=1.5, t2=45):
    z1, z2 = r1 * cis(t1), r2 * cis(t2)
    p, R, T = z1 * z2, r1 * r2, principal(t1 + t2)
    hit = abs(R - 1) < 1e-9 and T == 180
    fig = plt.figure(figsize=(11.5, 4.5))
    ax = fig.add_axes([0.03, 0.07, 0.36, 0.9]); plane(ax, LIM); ax.set_xticks(range(-3, 4)); ax.set_yticks(range(-3, 4))
    t = np.linspace(0, 2 * math.pi, 200); ax.plot(np.cos(t), np.sin(t), color=GHOST, lw=1.4, ls="--")
    ax.plot([-1], [0], "o", ms=22, mfc="none", mec=GOLD, mew=2.6, zorder=4)            # the target, −1
    ax.text(-1, -0.62, "−1", color=GOLD, fontsize=15, ha="center")
    arc(ax, 0.35, 0, t1, SKY, alpha=0.9); arc(ax, 0.5, 0, t2, MINT, alpha=0.9)
    arc(ax, 0.65, 0, t1 + t2, CORAL, alpha=0.9)
    off = R > LIM - 0.25                                                 # too long for the picture: drawn to the edge
    shown = p / R * (LIM - 0.25) if off else p
    arrow(ax, z1, SKY, lw=3); arrow(ax, z2, MINT, lw=3); arrow(ax, shown, GOLD if hit else CORAL, lw=4.2, z=6)
    s1 = -1 if t1 <= t2 else 1                                           # labels on the outer sides of the pair
    for z, lab, col, side, k in ((z1, "z₁", SKY, s1, 0.3), (z2, "z₂", MINT, -s1, 0.3),
                                 (shown, "z₁z₂", GOLD if hit else CORAL, -1.4, 0.45)):
        u = z / abs(z); v = u * 1j * side                                 # out along the arrow, and to one side
        ax.text(z.real + k * u.real + 0.28 * v.real, z.imag + k * u.imag + 0.28 * v.imag, lab, color=col,
                fontsize=15, ha="center", va="center", fontweight="bold")
    if off: ax.text(0, -LIM + 0.25, f"z₁z₂ runs off the edge (length {length(R)})", color=DIM, fontsize=12, ha="center")
    tp = fig.add_axes([0.46, 0.0, 0.54, 1.0]); tp.axis("off"); tp.set_xlim(0, 1); tp.set_ylim(0, 1)
    tp.text(0.0, 0.92, f"z₁ = {length(r1)}(cos {ang(t1)} + i sin {ang(t1)}) ≈ {cform(z1)}", color=SKY, fontsize=15)
    tp.text(0.0, 0.83, f"z₂ = {length(r2)}(cos {ang(t2)} + i sin {ang(t2)}) ≈ {cform(z2)}", color=MINT, fontsize=15)
    tp.text(0.0, 0.70, "z₁z₂: multiply the lengths, add the angles", color=DIM, fontsize=14)
    tp.text(0.0, 0.61, f"length  {length(r1)} × {length(r2)} = {length(R)}", color=CHALK, fontsize=18, family="monospace")
    tp.text(0.0, 0.52, f"angle   {ang(t1)} + {ang(t2)} = {ang(t1 + t2).strip('()')}", color=CHALK, fontsize=18, family="monospace")
    if T != t1 + t2: tp.text(0.0, 0.455, f"        the same as {ang(T).strip('()')}", color=DIM, fontsize=14, family="monospace")
    tp.text(0.0, 0.36, f"z₁z₂ ≈ {cform(p)}", color=GOLD if hit else CORAL, fontsize=24, fontweight="bold")
    if hit:
        tp.text(0.0, 0.24, "Exactly −1!", color=GOLD, fontsize=24, fontweight="bold")
        tp.text(0.0, 0.13, f"The lengths multiply to 1 and the angles add to 180°.", color=CHALK, fontsize=15)
    else:
        tp.text(0.0, 0.24, f"distance from −1: {abs(p + 1):.2f}", color=DIM, fontsize=16)
        tp.text(0.0, 0.13, "Aim for the gold ring at −1.", color=DIM, fontsize=15)
    plt.show()

R1 = FloatSlider(value=1.0, min=0.5, max=2.0, step=0.25, description="r₁")      # z₁: its length …
T1 = IntSlider(value=30, min=-180, max=180, step=15, description="θ₁ (°)")       # … and its angle
R2 = FloatSlider(value=1.5, min=0.5, max=2.0, step=0.25, description="r₂")      # z₂
T2 = IntSlider(value=45, min=-180, max=180, step=15, description="θ₂ (°)")

In [ ]:
interact(product, r1=R1, t1=T1, r2=R2, t2=T2);

In [ ]:
Z1, Z2 = (4, 70), (2, 25)                                            # (length, angle in degrees)
Q[5] = {"text": "z₁ = 4(cos 70° + i sin 70°) and z₂ = 2(cos 25° + i sin 25°). Give the modulus of z₁z₂ and its "
                "argument in degrees. Then write z₁ ÷ z₂ in the form a + bi (roots as sqrt(2), or two decimal places).",
        "answer": [Z1[0] * Z2[0], Z1[1] + Z2[1], sp.expand(sp.Rational(Z1[0], Z2[0]) * (sp.cos(pi / 4) + I * sp.sin(pi / 4)))],
        "tol": [1e-9, 1e-9, 0.006],
        "moves": ["multiply the lengths: 4 × 2.", "add the angles: 70° + 25°.",
                  "divide the lengths and subtract the angles, then cos 45° = sin 45° = √2/2."],
        "working": ["z₁z₂: lengths 4 × 2 = 8, angles 70° + 25° = 95°. So z₁z₂ = 8(cos 95° + i sin 95°).",
                    "z₁ ÷ z₂: lengths 4 ÷ 2 = 2, angles 70° − 25° = 45°. So z₁ ÷ z₂ = 2(cos 45° + i sin 45°).",
                    "cos 45° = sin 45° = √2/2, so z₁ ÷ z₂ = √2 + √2 i ≈ 1.41 + 1.41i."]}
assert Q[5]["answer"][:2] == [8, 95] and sp.simplify(Q[5]["answer"][2] - (sqrt(2) + sqrt(2) * I)) == 0
assert abs(4 * cis(70) / (2 * cis(25)) - (math.sqrt(2) + 1j * math.sqrt(2))) < 1e-12
practice(5)

## 6 · De Moivre's theorem

Multiplying by cos θ + i sin θ turns an arrow by θ without stretching it, so doing it n times turns it nθ:
(cos θ + i sin θ)ⁿ = cos nθ + i sin nθ. For a power of any complex number, put it in polar form, raise the length to
the power n and multiply the angle by n. Then take out full turns.

In [ ]:
Q[6] = {"text": "Use De Moivre's theorem to work out (1 + i√3)⁴ and (−1 + i)⁶, each in the form a + bi.",
        "answer": [sp.expand((1 + I * sqrt(3)) ** 4), sp.expand((-1 + I) ** 6)], "tol": [0.006, 1e-9],
        "moves": ["length 2, angle 60°: the answer has length 2⁴ and angle 4 × 60°.",
                  "length √2, angle 135°: the answer has length (√2)⁶ and angle 6 × 135°; take out full turns."],
        "working": ["1 + i√3 has length √(1 + 3) = 2 and angle 60° (π/3).",
                    "So (1 + i√3)⁴ has length 2⁴ = 16 and angle 4 × 60° = 240°.",
                    "cos 240° = −1/2 and sin 240° = −√3/2, so the answer is −8 − 8√3 i.",
                    "−1 + i has length √2 and angle 135° (3π/4).",
                    "So (−1 + i)⁶ has length (√2)⁶ = 8 and angle 6 × 135° = 810° = 2 × 360° + 90°.",
                    "90° is straight up, so the answer is 8i."]}
assert Q[6]["answer"] == [-8 - 8 * sqrt(3) * I, 8 * I] and 6 * 135 == 2 * 360 + 90
assert abs(16 * cis(240) - (-8 - 8 * math.sqrt(3) * 1j)) < 1e-12
practice(6)

## 7 · Roots of complex numbers

For the square root of a + bi, call it x + iy and match the parts: x² − y² = a and 2xy = b, and the modulus gives
x² + y² = |a + bi|. A complex number has n different n-th roots: take the n-th root of the length and spread the
angles 2π/n apart. The cube roots of unity are 1, ω = −1/2 + (√3/2)i and ω², with ω³ = 1 and 1 + ω + ω² = 0.

In [ ]:
w = sp.Rational(-1, 2) + sqrt(3) / 2 * I                           # ω
SQ7 = sp.solve(sp.Symbol("s") ** 2 - (5 - 12 * I), sp.Symbol("s"))
Q[7] = {"text": "Find the two square roots of 5 − 12i (in either order). Then work out ω¹⁰⁰ + ω²⁰⁰, where ω is the "
                "cube root of unity −1/2 + (√3/2)i.",
        "answer": [3 - 2 * I, -3 + 2 * I, sp.expand(w ** 100 + w ** 200)], "any_order": [(0, 1)],
        "moves": ["x² − y² = 5 and x² + y² = |5 − 12i| = 13; 2xy = −12 is negative, so x and y have opposite signs.",
                  "the other root is the first one turned half way round: its negative.",
                  "ω³ = 1, so take out threes: 100 = 3 × 33 + 1 and 200 = 3 × 66 + 2. Then 1 + ω + ω² = 0."],
        "working": ["Let (x + iy)² = 5 − 12i. Then x² − y² = 5 and 2xy = −12.",
                    "The modulus: x² + y² = |5 − 12i| = √(25 + 144) = 13.",
                    "Add: 2x² = 18, so x² = 9. Subtract: 2y² = 8, so y² = 4.",
                    "2xy = −12 is negative, so x and y have opposite signs: 3 − 2i and −3 + 2i.",
                    "Check: (3 − 2i)² = 9 − 12i + 4i² = 5 − 12i.",
                    "ω¹⁰⁰ = ω (100 = 3 × 33 + 1) and ω²⁰⁰ = ω² (200 = 3 × 66 + 2).",
                    "So ω¹⁰⁰ + ω²⁰⁰ = ω + ω² = −1, because 1 + ω + ω² = 0."]}
assert sorted(SQ7, key=str) == sorted([3 - 2 * I, -3 + 2 * I], key=str) and Q[7]["answer"][2] == -1
assert sp.expand(w ** 3) == 1 and sp.expand(1 + w + w ** 2) == 0
practice(7)

## 8 · Quadratic equations with complex roots

For ax² + bx + c = 0, x = (−b ± √(b² − 4ac)) / 2a. When the discriminant D = b² − 4ac is negative, √D = i√(−D), and
the roots are a conjugate pair, mirror images in the real axis. Working backwards, two roots that add to s and
multiply to p come from x² − sx + p = 0. Conjugate pairs need real coefficients.

In [ ]:
Q[8] = {"text": "Solve x² − 6x + 25 = 0 (two roots, in either order). Then find the quadratic equation in x, with "
                "real coefficients, that has −2 + 5i as a root.",
        "answer": [*sp.solve(x ** 2 - 6 * x + 25, x), sp.expand((x - (-2 + 5 * I)) * (x - (-2 - 5 * I)))],
        "any_order": [(0, 1)],
        "moves": ["D = 36 − 100 = −64, and √−64 = 8i.",
                  "the roots are (6 ± 8i) ÷ 2: a conjugate pair.",
                  "the other root is −2 − 5i. They add to −4 and multiply to 4 + 25."],
        "working": ["D = (−6)² − 4 × 1 × 25 = 36 − 100 = −64, so √D = 8i.",
                    "x = (6 ± 8i) ÷ 2 = 3 + 4i or 3 − 4i.",
                    "Check: they add to 6 and multiply to 9 + 16 = 25.",
                    "Real coefficients, so −2 − 5i is a root too. Sum: −4. Product: 4 + 25 = 29.",
                    "So the equation is x² − (−4)x + 29 = 0, which is x² + 4x + 29 = 0."]}
assert set(Q[8]["answer"][:2]) == {3 + 4 * I, 3 - 4 * I} and Q[8]["answer"][2] == x ** 2 + 4 * x + 29
assert same_equation(read("2x^2 + 8x + 58 = 0")[0], Q[8]["answer"][2]) and not same(read("x^2 - 4x + 29 = 0")[0], Q[8]["answer"][2])
practice(8)

## 9 · Euler's formula

Euler's formula: e^(iθ) = cos θ + i sin θ, the arrow of length one at angle θ. Put θ = π and e^(iπ) = −1, so
e^(iπ) + 1 = 0. Any complex number is r e^(iθ), its exponential form: multiply the r's and add the angles, as with
any powers. And cos θ = (e^(iθ) + e^(−iθ))/2, sin θ = (e^(iθ) − e^(−iθ))/2i.

In [ ]:
Q[9] = {"text": "Write 2e^(iπ/6) in the form a + bi. Then write −4i in exponential form r e^(iθ): give r and the "
                "principal argument θ (in radians).",
        "answer": [sp.expand_complex(2 * sp.exp(I * pi / 6)), sp.Abs(-4 * I), sp.arg(-4 * I)],
        "tol": [0.006, 1e-9, 0.006],
        "moves": ["2(cos π/6 + i sin π/6), with cos π/6 = √3/2 and sin π/6 = 1/2.",
                  "the length of −4i: how far it is from zero.",
                  "−4i points straight down."],
        "working": ["2e^(iπ/6) = 2(cos π/6 + i sin π/6) = 2(√3/2 + i/2) = √3 + i ≈ 1.73 + i.",
                    "−4i is 4 from zero, straight down: r = 4 and θ = −π/2.",
                    "So −4i = 4e^(−iπ/2). Check: 4(cos(−π/2) + i sin(−π/2)) = 4(0 − i) = −4i."]}
assert Q[9]["answer"] == [sqrt(3) + I, 4, -pi / 2] and abs(4 * cmath.exp(-1j * math.pi / 2) + 4j) < 1e-12
assert all(same(read(s)[0], Q[9]["answer"][0], 0.006) for s in ("sqrt(3) + i", "1.73 + i", "2e^(i pi/6)", "√3+i"))
practice(9)

## Sources
NCERT Mathematics Class XI, ch. 4 (Complex Numbers and Quadratic Equations); OpenStax, Precalculus 2e, 3.1 (Complex
Numbers) and 8.5 (Polar Form of Complex Numbers); Pearson Edexcel A level Further Mathematics (9FM0), Core Pure,
topic 2 (Complex numbers). All retrieved 2026-10-31. Every question is our own.